# Trash to Trust: run the whole paper

SUSCON XIII full paper, A. Kundu, Vinod Gupta School of Management, IIT Kharagpur.

**Runtime → Run all.** This notebook downloads the code and the dataset, installs the exact pinned environment, regenerates every table behind the paper (about 6–8 minutes on Colab), and checks every claim. It ends with the verification report.

No Google Drive access or sign-in is needed. The dataset ships inside the repository.

In [ ]:
import os, subprocess, sys
REPO = 'https://github.com/Githubakash123/trash-to-trust-suscon2026.git'
if os.path.exists('reproduce.py'):
    pass
elif os.path.exists('../reproduce.py'):
    os.chdir('..')
else:
    if not os.path.exists('trash-to-trust-suscon2026'):
        subprocess.run(['git', 'clone', '--depth', '1', REPO], check=True)
    os.chdir('trash-to-trust-suscon2026')
print('working in', os.getcwd())

## 1. Install the pinned environment
On Colab, the packages go into a separate virtual environment created with `uv`, so Colab's preinstalled versions don't interfere. Elsewhere (Binder, local Jupyter), the current interpreter is used.

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
# On Colab (or when T2T_VENV is set, as in this repo's CI) use an isolated uv environment.
VENV = '/content/t2t' if IN_COLAB else os.environ.get('T2T_VENV')
if VENV:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '-q', '--python', '3.12', VENV], check=True)
    PY = os.path.join(VENV, 'bin', 'python')
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '-q', '--python', PY, '-r', 'requirements.txt'], check=True)
else:
    PY = sys.executable
    subprocess.run([PY, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('python:', subprocess.run([PY, '--version'], capture_output=True, text=True).stdout.strip())

## 2. Confirm the code and data are the paper's files

In [ ]:
r = subprocess.run([PY, 'verification/check_provenance.py'])
assert r.returncode == 0

## 3. Regenerate everything and verify every claim (about 6–8 min)

In [ ]:
r = subprocess.run([PY, 'reproduce.py'])
assert r.returncode == 0, 'verification FAILED - see the report below'

## 4. The verification report
Statuses: **REPRODUCED**, **REPRODUCED_WITH_CORRECTION** (the number regenerates, but the paper's unit or description is wrong), **LEGACY_ONLY** (only an old Colab output holds the number) and **NOT_SUPPORTED** (the code does not do what the paper says). See `ERRATA.md` for each correction.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('VERIFICATION_REPORT.md', encoding='utf-8').read()))

## 5. Download all regenerated outputs (optional)

In [ ]:
import shutil
zip_path = shutil.make_archive('trash_to_trust_outputs', 'zip', '.', 'outputs')
print(zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)